# Entrainer Effect on Acetone-Methanol VLE

This notebook evaluates the effect of adding an entrainer (ethyl formate) on the acetone-methanol VLE by calculating ternary equilibrium at different fixed entrainer compositions and plotting pseudo-binary VLE curves.

In [ ]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

# import the necessary packages
from src.molecules import StructuralValidator
from src.pure import PropertyCalculator
from src.pure import AntoineCalculator
from src.data import load_unifac_data
from src.mixture.unifac.pyomo_vle import pyo_vle
import numpy as np
import matplotlib.pyplot as plt

## Step 1: Define Compounds

Define the ternary mixture:
- Component A: Acetone (more volatile)
- Component B: Methanol (less volatile)
- Component E

In [ ]:
# Define the compounds with UNIFAC groups
mixture = {
    'acetone': {'CH3': 1, 'CH3CO': 1},
    'methanol': {'CH3OH': 1},
    'entrainer': {'CH2': 1, 'HCO': 2}
}

# Validate the structures
validator = StructuralValidator()
validator.validate(mixture)

print("Compounds defined and validated:")
for name, groups in mixture.items():
    print(f"  {name}: {groups}")

## Step 2: Calculate Pure Component Properties

Calculate critical properties, boiling point, acentric factor, and heat of vaporization using the Constantinou-Gani method.

In [ ]:
# Calculate pure component properties
prop_list = ['tc', 'pc', 'tb', 'ac', 'hv']
pure_calc = PropertyCalculator()
pure_props = pure_calc.calculate(mixture, prop_list)

# Override acetone properties with experimental values for better accuracy
pure_props['acetone']['tc'] = 508.1  # K
pure_props['acetone']['pc'] = 48.0   # bar
pure_props['acetone']['tb'] = 329.3  # K

print("Pure component properties:")
for compound, props in pure_props.items():
    print(f"\n{compound}:")
    for prop, value in props.items():
        print(f"  {prop} = {value:.2f}")

## Step 3: Calculate Antoine Parameters

Calculate Antoine equation parameters (A, B, C) for vapor pressure using the enthalpy of vaporization method.

In [ ]:
# Calculate Antoine coefficients
antoine_params = {}
antoine_calc = AntoineCalculator()

for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(
        mixture, critical_properties=props, method='enthalpy_vap'
    )
    
    print(f"{compound}: optimization status = {results['termination_condition']}")
    
    antoine_params[compound] = {
        'A': results['A'],
        'B': results['B'],
        'C': results['C']
    }

print("\nAntoine parameters:")
for compound, params in antoine_params.items():
    print(f"\n{compound}:")
    print(f"  A = {params['A']:.4f}")
    print(f"  B = {params['B']:.4f}")
    print(f"  C = {params['C']:.4f}")

## Step 4: Setup Ternary VLE Calculation Parameters

Define the range of entrainer compositions to test and number of data points for each case.

In [ ]:
# Setup for ternary VLE calculations at different entrainer compositions
n_data = 50  # Number of data points for each entrainer concentration
n_entrainer = 4  # Number of different entrainer concentrations to test

# Define entrainer mass fractions to test (including 0 for baseline)
x_E_values = np.array([0.0, 0.1, 0.2, 0.3])

print(f"Testing {n_entrainer} different entrainer concentrations:")
print(f"x_E = {x_E_values}")
print(f"Number of composition points per entrainer level: {n_data}")
print(f"Total VLE calculations: {n_entrainer * n_data}")

## Step 5: Generate Compositions and Perform VLE Calculations

For each fixed entrainer composition:
- Generate compositions where acetone varies from 0 to (1 - x_E)
- Methanol fills the remainder: x_methanol = 1 - x_acetone - x_entrainer
- Stack all compositions and solve VLE in batch
- Store results in 3D arrays [entrainer_level, data_point, component]

In [ ]:
# Load UNIFAC parameters
rk_data, qk_data, interaction_data = load_unifac_data()

# Extract boiling points for temperature initialization
boiling_points = [props['tb'] for props in pure_props.values()]
T_min = min(boiling_points)
T_max = max(boiling_points)

print(f"Temperature range for initialization: {T_min:.1f} - {T_max:.1f} K")

# Generate all compositions for all entrainer levels
all_compositions = []
all_T_guess = []

for i, x_E in enumerate(x_E_values):
    # Generate composition grid for this entrainer concentration
    # Component A (acetone) varies from 0 to (1 - x_E)
    x_A_range = np.linspace(0, 1 - x_E, n_data)
    x_B_range = 1 - x_E - x_A_range  # Component B (methanol) is the remainder
    x_E_array = np.full(n_data, x_E)  # Entrainer is fixed
    
    # Combine into composition array [x_acetone, x_methanol, x_entrainer]
    x_compositions = np.column_stack([x_A_range, x_B_range, x_E_array])
    all_compositions.append(x_compositions)
    
    # Temperature guess (interpolate between boiling points)
    T_guess = np.linspace(T_min, T_max, n_data)
    all_T_guess.append(T_guess)

# Stack all compositions for batch calculation
x_all = np.vstack(all_compositions)
T_guess_all = np.concatenate(all_T_guess)

print(f"\nGenerated composition array shape: {x_all.shape}")
print(f"Total number of VLE calculations: {len(x_all)}")

# Perform VLE calculation for all compositions at once
print("\nPerforming ternary VLE calculations...")
try:
    results, model = pyo_vle(x_all, T_guess_all, mixture,
                            rk_data, qk_data, interaction_data,
                            antoine_params, verbose=False)
    
    print("✓ VLE calculations successful!")
    
except Exception as e:
    print(f"✗ VLE calculations failed: {str(e)}")
    raise

## Step 6: Extract Results into 3D Arrays

Reshape results into 3D arrays with dimensions:
- Dimension 0: Entrainer concentration level (0 to n_entrainer-1)
- Dimension 1: Data point within each level (0 to n_data-1)
- Dimension 2: Component (0=acetone, 1=methanol, 2=entrainer)

In [ ]:
# Verify dimensions before extracting results
print(f"Verifying dimensions:")
print(f"  x_all shape: {x_all.shape}")
print(f"  Expected: ({n_entrainer * n_data}, 3) = ({n_entrainer}×{n_data}, 3)")

# Use actual dimensions from x_all for robustness
n_total_actual = x_all.shape[0]
n_components = x_all.shape[1]

# Verify consistency
expected_total = n_entrainer * n_data

# Initialize arrays to store results
n_total = n_total_actual
y_all = np.zeros((n_total, n_components))
T_all = np.zeros(n_total)
gamma_all = np.zeros((n_total, n_components))

# Extract results from Pyomo model
for i in range(n_total):
    T_all[i] = model.T[i].value if model.T[i].value else T_guess_all[i]
    
    for k in range(n_components):
        y_all[i, k] = model.y[i, k].value
        gamma_all[i, k] = model.gamma[i, k].value

# Reshape into 3D arrays [entrainer_level, data_point, component]
x_liquid_3d = x_all.reshape(n_entrainer, n_data, n_components)
y_vapor_3d = y_all.reshape(n_entrainer, n_data, n_components)
T_bubble_3d = T_all.reshape(n_entrainer, n_data)
gamma_3d = gamma_all.reshape(n_entrainer, n_data, n_components)

print("\nResults extracted and reshaped:")
print(f"  x_liquid_3d shape: {x_liquid_3d.shape}")
print(f"  y_vapor_3d shape: {y_vapor_3d.shape}")
print(f"  T_bubble_3d shape: {T_bubble_3d.shape}")
print(f"  gamma_3d shape: {gamma_3d.shape}")

# Display temperature range for each entrainer concentration
print("\nBubble point temperature ranges:")
for i, x_E in enumerate(x_E_values):
    T_min_i = np.min(T_bubble_3d[i, :])
    T_max_i = np.max(T_bubble_3d[i, :])
    print(f"  x_E = {x_E:.1f}: T = {T_min_i:.1f} - {T_max_i:.1f} K")

## Step 7: Calculate Entrainer-Free Basis Compositions

To visualize the entrainer effect, convert compositions to an entrainer-free basis:

$$x'_A = \frac{x_A}{x_A + x_B} = \frac{x_A}{1 - x_E}$$

$$y'_A = \frac{y_A}{y_A + y_B} = \frac{y_A}{1 - y_E}$$

This allows comparison of acetone-methanol equilibrium at different entrainer concentrations.

In [ ]:
# Calculate entrainer-free basis compositions
x_prime_3d = np.zeros((n_entrainer, n_data))
y_prime_3d = np.zeros((n_entrainer, n_data))

for i in range(n_entrainer):
    # Liquid phase entrainer-free basis
    x_A = x_liquid_3d[i, :, 0]
    x_B = x_liquid_3d[i, :, 1]
    x_sum_AB = x_A + x_B
    
    # Avoid division by zero
    x_prime_3d[i, :] = np.where(x_sum_AB > 1e-10, x_A / x_sum_AB, 0)
    
    # Vapor phase entrainer-free basis
    y_A = y_vapor_3d[i, :, 0]
    y_B = y_vapor_3d[i, :, 1]
    y_sum_AB = y_A + y_B
    
    # Avoid division by zero
    y_prime_3d[i, :] = np.where(y_sum_AB > 1e-10, y_A / y_sum_AB, 0)

print("Entrainer-free basis compositions calculated")
print(f"  x' shape: {x_prime_3d.shape}")
print(f"  y' shape: {y_prime_3d.shape}")

# Display sample values
print("\nSample entrainer-free compositions (midpoint):")
mid_idx = n_data // 2
for i, x_E in enumerate(x_E_values):
    print(f"  x_E = {x_E:.1f}: x'_acetone = {x_prime_3d[i, mid_idx]:.3f}, y'_acetone = {y_prime_3d[i, mid_idx]:.3f}")

## Step 8: Visualize Entrainer Effect - x-y Diagram

Plot pseudo-binary x-y diagrams showing how different entrainer concentrations affect the acetone-methanol equilibrium.

In [ ]:
# Publication-quality x-y diagram showing entrainer effect
fig, ax = plt.subplots(figsize=(7, 7), dpi=300)

# Plot diagonal reference line
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.6, label='$y = x$', zorder=1)

# Define colors for different entrainer concentrations (colormap)
colors = plt.cm.viridis(np.linspace(0, 0.9, n_entrainer))

# Plot VLE curves for each entrainer concentration
for i, x_E in enumerate(x_E_values):
    label = f'$x_E = {x_E:.1f}$' if x_E > 0 else 'No entrainer'
    ax.plot(x_prime_3d[i, :], y_prime_3d[i, :], 
            color=colors[i], linewidth=2.5, label=label, zorder=3)

# Formatting
ax.set_xlabel('Liquid mole fraction (entrainer-free), $x\'_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')
ax.set_ylabel('Vapor mole fraction (entrainer-free), $y\'_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')
ax.set_title('Entrainer Effect on Acetone-Methanol VLE', fontsize=14, fontweight='bold', pad=15)

# Set equal aspect ratio and limits
ax.set_aspect('equal')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# Tick formatting
ax.tick_params(axis='both', which='major', labelsize=11, direction='in',
               length=6, width=1.5, top=True, right=True)

# Legend
ax.legend(loc='upper left', fontsize=10.5, frameon=True, fancybox=False,
          edgecolor='black', framealpha=1)

# Grid
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.3, color='gray')

plt.tight_layout()

# Save figure
plt.savefig('entrainer_effect_xy.pdf', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none', format='pdf')
plt.savefig('entrainer_effect_xy.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none', format='png')

print("Figure saved as:")
print("  - entrainer_effect_xy.pdf (for LaTeX)")
print("  - entrainer_effect_xy.png (for presentations)")

plt.show()

In [ ]:
# Zoomed-in x-y diagram (0.9 to 1.0 range)
fig, ax = plt.subplots(figsize=(7, 7), dpi=300)

# Plot diagonal reference line
ax.plot([0.9, 1.0], [0.9, 1.0], 'k--', linewidth=1.5, alpha=0.6, label='$y = x$', zorder=1)

# Plot VLE curves for each entrainer concentration
for i, x_E in enumerate(x_E_values):
    # Filter data to zoom range
    mask = x_prime_3d[i, :] >= 0.9
    x_zoom = x_prime_3d[i, mask]
    y_zoom = y_prime_3d[i, mask]
    
    label = f'$x_E = {x_E:.1f}$' if x_E > 0 else 'No entrainer'
    ax.plot(x_zoom, y_zoom, 
            color=colors[i], linewidth=2.5, label=label, zorder=3)

# Formatting
ax.set_xlabel('Liquid mole fraction (entrainer-free), $x\'_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')
ax.set_ylabel('Vapor mole fraction (entrainer-free), $y\'_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')
ax.set_title('Entrainer Effect (Zoomed: 0.9-1.0)', fontsize=14, fontweight='bold', pad=15)

# Set equal aspect ratio and zoomed limits
ax.set_aspect('equal')
ax.set_xlim(0.9, 1.0)
ax.set_ylim(0.9, 1.0)

# Tick formatting
ax.tick_params(axis='both', which='major', labelsize=11, direction='in',
               length=6, width=1.5, top=True, right=True)

# Legend
ax.legend(loc='lower right', fontsize=10.5, frameon=True, fancybox=False,
          edgecolor='black', framealpha=1)

# Grid
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.3, color='gray')

plt.tight_layout()

# Save figure
plt.savefig('entrainer_effect_xy_zoom.pdf', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none', format='pdf')
plt.savefig('entrainer_effect_xy_zoom.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none', format='png')

print("Zoomed figure saved as:")
print("  - entrainer_effect_xy_zoom.pdf (for LaTeX)")
print("  - entrainer_effect_xy_zoom.png (for presentations)")

plt.show()

## Step 9: Calculate and Visualize Relative Volatility

Calculate relative volatility to quantify the separation ease:

$$\alpha_{AB} = \frac{y_A / x_A}{y_B / x_B}$$

When α > 1, separation is easier. The azeotrope occurs when α = 1.

In [ ]:
# Calculate relative volatility for each entrainer concentration
alpha_AB = np.zeros((n_entrainer, n_data))

for i in range(n_entrainer):
    x_A = x_liquid_3d[i, :, 0]
    x_B = x_liquid_3d[i, :, 1]
    y_A = y_vapor_3d[i, :, 0]
    y_B = y_vapor_3d[i, :, 1]
    
    # Avoid division by zero
    mask = (x_A > 1e-10) & (x_B > 1e-10) & (y_A > 1e-10) & (y_B > 1e-10)
    alpha_AB[i, mask] = (y_A[mask] / x_A[mask]) / (y_B[mask] / x_B[mask])
    alpha_AB[i, ~mask] = np.nan

print("Relative volatility calculated")
print(f"  α_AB shape: {alpha_AB.shape}")

# Display statistics
print("\nRelative volatility statistics:")
for i, x_E in enumerate(x_E_values):
    alpha_min = np.nanmin(alpha_AB[i, :])
    alpha_max = np.nanmax(alpha_AB[i, :])
    alpha_mean = np.nanmean(alpha_AB[i, :])
    print(f"  x_E = {x_E:.1f}: α_min = {alpha_min:.2f}, α_max = {alpha_max:.2f}, α_mean = {alpha_mean:.2f}")

In [ ]:
# Combined figure: x-y diagram and relative volatility as subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)

# ========== Left panel: x-y diagram ==========
# Plot diagonal reference line
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.6, label='$y = x$', zorder=1)

# Plot VLE curves for each entrainer concentration
for i, x_E in enumerate(x_E_values):
    label = f'$x_E = {x_E:.1f}$' if x_E > 0 else 'No entrainer'
    ax1.plot(x_prime_3d[i, :], y_prime_3d[i, :], 
            color=colors[i], linewidth=2.5, label=label, zorder=3)

# Formatting for left panel
ax1.set_xlabel('Liquid mole fraction (entrainer-free), $x\'_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')
ax1.set_ylabel('Vapor mole fraction (entrainer-free), $y\'_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')

# Set equal aspect ratio and limits
ax1.set_aspect('equal')
ax1.set_xlim(-0.02, 1.02)
ax1.set_ylim(-0.02, 1.02)

# Tick formatting
ax1.tick_params(axis='both', which='major', labelsize=11, direction='in',
               length=6, width=1.5, top=True, right=True)

# Legend
ax1.legend(loc='upper left', fontsize=10, frameon=True, fancybox=False,
          edgecolor='black', framealpha=1)

# Grid
ax1.grid(True, linestyle='--', linewidth=0.5, alpha=0.3, color='gray')


# ========== Right panel: Relative volatility ==========
for i, x_E in enumerate(x_E_values):
    label = f'$x_E = {x_E:.1f}$' if x_E > 0 else 'No entrainer'
    ax2.plot(x_prime_3d[i, :], alpha_AB[i, :],
            color=colors[i], linewidth=2.5, label=label)

# Formatting for right panel
ax2.set_xlabel('Liquid mole fraction (entrainer-free), $x\'_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')
ax2.set_ylabel('Relative Volatility, $\\alpha_{AB}$ [-]',
              fontsize=13, fontweight='normal')

# Add reference line at α = 1 (azeotrope)
ax2.axhline(1, color='red', linestyle='--', linewidth=1.5, alpha=0.6, 
           label='$\\alpha = 1$ (azeotrope)', zorder=1)

ax2.set_xlim(-0.02, 1.02)
ax2.tick_params(axis='both', which='major', labelsize=11, direction='in',
               length=6, width=1.5, top=True, right=True)

ax2.legend(loc='best', fontsize=10, frameon=True, fancybox=False,
          edgecolor='black', framealpha=1)

ax2.grid(True, linestyle='--', linewidth=0.5, alpha=0.3, color='gray')

# Tighten spacing between subplots
plt.subplots_adjust(wspace=0.15)

# Save combined figure
plt.savefig('entrainer_effect_combined.pdf', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none', format='pdf')
plt.savefig('entrainer_effect_combined.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none', format='png')

print("Combined figure saved as:")
print("  - entrainer_effect_combined.pdf (for LaTeX)")
print("  - entrainer_effect_combined.png (for presentations)")

plt.show()

In [ ]:
# Plot relative volatility vs composition
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=300)

for i, x_E in enumerate(x_E_values):
    label = f'$x_E = {x_E:.1f}$' if x_E > 0 else 'No entrainer'
    ax.plot(x_prime_3d[i, :], alpha_AB[i, :],
            color=colors[i], linewidth=2.5, label=label)

# Formatting
ax.set_xlabel('Liquid mole fraction (entrainer-free), $x\'_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')
ax.set_ylabel('Relative Volatility, $\\alpha_{AB}$ [-]',
              fontsize=13, fontweight='normal')
ax.set_title('Effect of Entrainer on Relative Volatility', fontsize=14, fontweight='bold', pad=15)

# Add reference line at α = 1 (azeotrope)
ax.axhline(1, color='red', linestyle='--', linewidth=1.5, alpha=0.6, label='$\\alpha = 1$ (azeotrope)')

ax.set_xlim(-0.02, 1.02)
ax.tick_params(axis='both', which='major', labelsize=11, direction='in',
               length=6, width=1.5, top=True, right=True)

ax.legend(loc='best', fontsize=10.5, frameon=True, fancybox=False,
          edgecolor='black', framealpha=1)

ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.3, color='gray')

plt.tight_layout()

plt.savefig('entrainer_effect_alpha.pdf', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none', format='pdf')
plt.savefig('entrainer_effect_alpha.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none', format='png')

print("Figure saved as:")
print("  - entrainer_effect_alpha.pdf (for LaTeX)")
print("  - entrainer_effect_alpha.png (for presentations)")

plt.show()